# dgx-spark-hijinks: Gemma 3+4 Triton-retirement benchmark (Colab G4 / RTX PRO 6000, sm_120)

**Version: VICUNA.** Proves the Triton retirement with numbers, decomposed so a vLLM
maintainer cant ask "which variable helped?". Per model, three cells:

| cell | backend | KV | isolates |
|---|---|---|---|
| **A** | TRITON_ATTN | bf16 | the slow fallback baseline |
| **B** | FLASHINFER + VO-split | bf16 | **A->B = the retirement** (same dtype, backend only) |
| **C** | FLASHINFER + VO-split | **nvfp4** | **B->C = the 4-bit KV win** (same backend, dtype only) |

**Triton/nvfp4 is N/A by construction** - Triton has no NVFP4 KV path. That empty cell is
the necessity proof: shipping nvfp4 KV *requires* FlashInfer.

**How to run:** open this notebook in **3 Colab G4 runtimes**; set `CELL = "A" | "B" | "C"`
(matrix cell) differently in each; Run-All. Each writes to
`Drive/gemma_retirement_bench/<cell>/` so the three converge; hand the folder back and
Claude assembles the A/B/C table.

Wheel: `sm120a-wheels-70ab8b243` (has the Gemma 4 retirement + VO-split + nvfp4 KV).


In [ ]:
# Cell 1 - runtime guard. REQUIRE compute capability 12.x; fail fast otherwise.
NOTEBOOK_VERSION = 'VICUNA'  # KEEP IN SYNC with the banner in the title cell
import subprocess
q = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total,compute_cap',
                    '--format=csv,noheader'], capture_output=True, text=True)
assert q.returncode == 0 and q.stdout.strip(), (
    'nvidia-smi failed - no GPU attached.\n'
    'Runtime -> Change runtime type -> select the G4 GPU '
    '(NVIDIA RTX PRO 6000 Blackwell), then re-run this cell.')
name, driver, mem, cc = [s.strip() for s in q.stdout.strip().splitlines()[0].split(',')]
print(f'GPU: {name} | driver {driver} | {mem} | compute capability {cc}')
assert int(cc.split('.')[0]) == 12, (
    f'Got {name} (CC {cc}) - this notebook is sm_120a-specific and is MEANINGLESS on '
    'A100/L4/T4 runtimes.\nRuntime -> Change runtime type -> pick the G4 runtime '
    '(NVIDIA RTX PRO 6000 Blackwell, CC 12.0, 96 GB), then restart.')
VRAM_GIB = float(mem.split()[0]) / 1024.0
print(f'VRAM: {VRAM_GIB:.1f} GiB ' + ('(G4-class 96 GB: full Gemma 4 ladder fits)'
      if VRAM_GIB >= 80 else '(consumer-class: model menu trimmed below)'))

ALL_MODELS = ['google/gemma-4-E2B-it', 'google/gemma-4-E4B-it', 'google/gemma-4-12B-it',
              'google/gemma-4-26B-A4B-it', 'google/gemma-4-31B-it']
if VRAM_GIB >= 80:
    MODEL_MENU = list(ALL_MODELS)          # 96 GB G4: full ladder incl. 26B-A4B MoE + 31B
elif VRAM_GIB >= 40:
    MODEL_MENU = ALL_MODELS[:3]            # 48 GB class: E2B + E4B + 12B
else:
    MODEL_MENU = ALL_MODELS[:2]            # 16 GB consumer Blackwell: E2B + E4B
print('model menu for this box:', MODEL_MENU)


In [ ]:
# Cell 2 - WHOLESALE environment (the campaign's hard-won recipe; deterministic, no
# iterating). ~10-15 min on a fresh runtime, dominated by the apt CUDA toolkit + wheel.
#
# ============================== EDIT ME ======================================
WHEEL_RELEASE_TAG = 'sm120a-wheels-70ab8b243'   # latest green tag from
#       https://github.com/jethac/vllm/releases  (built by build-sm120a-wheel.yml;
#       tag = sm120a-wheels-<9-char shortsha of spark/hijinks-e2-vllm>)
# =============================================================================
VLLM_REPO = 'jethac/vllm'
FLASHINFER_BRANCH = 'spark/hijinks-022-fa2-d512'   # source-tree, runtime JIT (no wheel
#       on purpose: JIT compiles only the kernels actually used, keyed to this GPU;
#       known-good with the apt cuda-toolkit on Colab since notebook DINGO)
HIJINKS_BRANCH = 'epoch2'
TORCH_PIN = 'torch==2.12.0'                         # P520 sm_120 known-good ABI (cu130)
VENV = '/content/g4venv'
import json as _json, os, subprocess, sys, urllib.request

def sh(cmd, check=True, env=None):
    print('+', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    if r.stdout: print(r.stdout[-2000:])
    if r.stderr: print(r.stderr[-2000:])
    if check and r.returncode != 0:
        raise RuntimeError(f'failed ({r.returncode}): {cmd}')
    return r

# 1) full nvcc 13 via NVIDIA apt repo. Wholesale cuda-toolkit-13-0, NOT a minimal
#    package set - the minimal set cost five round-trips (nvcc, nvrtc, cusparse,
#    cublas, ...). Verified dead end: the pip cu13 'nvcc' wheel is ptxas-only.
def _have_nvcc():
    try:
        return subprocess.run(['/usr/local/cuda-13.0/bin/nvcc', '--version'],
                              capture_output=True).returncode == 0
    except FileNotFoundError:
        return False
if not _have_nvcc():
    ub = '2404' if '24.04' in open('/etc/os-release').read() else '2204'
    sh(f'wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu{ub}/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/ck.deb')
    sh('dpkg -i /tmp/ck.deb && apt-get -qq update')
    sh('apt-get -qq install -y cuda-toolkit-13-0')   # ~3 GB / ~8 min, deterministic
    sh('apt-get -qq install -y ninja-build')  # FlashInfer JIT needs ninja on PATH
CUDA_HOME = '/usr/local/cuda-13.0'
sh(f'{CUDA_HOME}/bin/nvcc --version | tail -1')

# 2) fresh venv - isolates the campaign stack from Colab's preinstalled torch/vllm.
#    (No kernel restart needed: serving runs as venv subprocesses, not in this kernel.)
# virtualenv, not `python -m venv`: Colab's python breaks venv's ensurepip
# bootstrap (returns 1); virtualenv ships its own pip and dodges it.
if not os.path.isfile(f'{VENV}/bin/pip'):
    sh(f'rm -rf {VENV}', check=False)
    sh(f'{sys.executable} -m pip install -q virtualenv')
    sh(f'{sys.executable} -m virtualenv {VENV}')
VPIP = f'{VENV}/bin/pip'
VPY = f'{VENV}/bin/python'
sh(f'{VPIP} install -q --upgrade pip')

# 3) torch cu130, pinned (must go in BEFORE the vLLM wheel so its deps resolve
#    against this torch, not a PyPI cu126 one).
# torch + torchvision together from the SAME cu130 index so their C++ ABIs match.
# torchvision is required: Gemma 4's multimodal processor imports it, and vLLM
# inspects the multimodal arch even under --language-model-only.
sh(f'{VPIP} install -q {TORCH_PIN} torchvision --index-url https://download.pytorch.org/whl/cu130')

# 4) the CI-built sm_120a vLLM wheel from the GitHub Release (THE point of this lane:
#    this cell replaces the 1-2 h Colab source build).
rel = _json.load(urllib.request.urlopen(
    f'https://api.github.com/repos/{VLLM_REPO}/releases/tags/{WHEEL_RELEASE_TAG}'))
whls = [a['browser_download_url'] for a in rel['assets'] if a['name'].endswith('.whl')]
assert whls, (f'no .whl asset on release {WHEEL_RELEASE_TAG} - check '
              f'https://github.com/{VLLM_REPO}/releases  (CI still running? wrong tag?) '
              'Last resort: the SLOW source-build fallback cell below.')
WHEEL_URL = whls[0]
print('wheel:', WHEEL_URL)
sh(f'{VPIP} install -q "{WHEEL_URL}"')
# campaign rule: no packaged FlashInfer alongside the source tree (stale cubin/JIT
# packages were convicted before - WHEEL_CONTAINER_MATRIX r7 post-mortem).
sh(f'{VPIP} uninstall -q -y flashinfer-python flashinfer-jit-cache flashinfer-cubin',
   check=False)

# 5) FlashInfer source tree (campaign branch) on PYTHONPATH, JIT mode + its deps.
if not os.path.isdir('/content/flashinfer'):
    sh(f'git clone --recursive https://github.com/jethac/flashinfer -b {FLASHINFER_BRANCH} /content/flashinfer')
# FlashInfer JIT reads kernel sources from flashinfer/data/, which a raw clone
# (vs pip install) leaves empty. Symlink the source trees in so the JIT can build.
sh('mkdir -p /content/flashinfer/flashinfer/data')
for _tgt, _name in [('../../csrc','csrc'), ('../../include','include'),
                    ('../../3rdparty/cutlass','cutlass'), ('../../3rdparty/spdlog','spdlog'),
                    ('../../3rdparty/cccl','cccl'), ('../../3rdparty/nccl','nccl'),
                    ('../../3rdparty/nixl','nixl')]:
    sh(f'ln -sfn {_tgt} /content/flashinfer/flashinfer/data/{_name}')
sh(f'{VPIP} install -q ninja jinja2 packaging apache-tvm-ffi filelock requests einops nvidia-ml-py')

# 5b) transformers pin: Gemma 4 (Gemma4ForConditionalGeneration / gemma4_unified)
#     needs a transformers new enough to inspect the arch; vLLM's dep resolve
#     pulls an older one. 5.11.0 is the campaign-proven version (Spark r10).
sh(f'{VPIP} install -q transformers==5.11.0')

# 6) campaign repo: serving/PPL harnesses + the bundled PPL corpus slices.
if not os.path.isdir('/content/dgx-spark-hijinks'):
    sh(f'git clone --depth 5 https://github.com/jethac/dgx-spark-hijinks -b {HIJINKS_BRANCH} /content/dgx-spark-hijinks')
HIJINKS = '/content/dgx-spark-hijinks'
C1_CORPUS = f'{HIJINKS}/results/claude_anomaly_corpus_sweep_20260611/docs/c1_ppl_corpus.md'
assert os.path.isfile(C1_CORPUS), 'bundled C1 corpus slice missing from clone'

# 7) import gate (this box HAS the GPU, unlike the CI runner - safe to import).
ENV_BASE = dict(os.environ,
                CUDA_HOME=CUDA_HOME,
                PATH=f'{VENV}/bin:{CUDA_HOME}/bin:' + os.environ['PATH'],
                PYTHONPATH='/content/flashinfer')
r = sh(f'{VPY} -c "import torch, vllm, flashinfer; '
       'print(\'torch\', torch.__version__, \'| vllm\', vllm.__version__, '
       '\'| flashinfer\', flashinfer.__version__)"', env=ENV_BASE)
print('ENVIRONMENT READY')


<details><summary><b>Fallback: full source build (SLOW, ~1-2 h GPU-time wasted - only
when no release wheel matches)</b></summary>

The cell below rebuilds the sm_120a wheel from source on this runtime. It exists ONLY
for the gap where `spark/hijinks-e2-vllm` moved and CI hasn't published the new tag yet.
Set `RUN_SOURCE_BUILD = True` explicitly; otherwise it refuses to run. Prefer waiting
for CI or re-running with the previous green tag.
</details>


In [ ]:
# FALLBACK (SLOW) - source-build the sm_120a wheel on this runtime. Normally SKIP.
RUN_SOURCE_BUILD = False   # <- flip deliberately; this burns 1-2 h of paid GPU runtime
if not RUN_SOURCE_BUILD:
    print('skipped (RUN_SOURCE_BUILD=False) - using the CI release wheel. Good.')
else:
    import os, subprocess
    sh('apt-get -qq install -y ccache || true', check=False)
    if not os.path.isdir('/content/vllm-src'):
        sh('git clone --depth 50 https://github.com/jethac/vllm -b spark/hijinks-e2-vllm /content/vllm-src')
    # campaign use-existing-torch pattern (P520 build_vllm_hijinks.sh):
    sh(f'cd /content/vllm-src && {VPY} use_existing_torch.py')
    sh(f'{VPIP} install -q -r /content/vllm-src/requirements/build/cuda.txt')
    env = dict(ENV_BASE, TORCH_CUDA_ARCH_LIST='12.0a', VLLM_MAIN_CUDA_VERSION='13.0',
               MAX_JOBS='4', NVCC_THREADS='1')
    sh(f'cd /content/vllm-src && {VPIP} wheel --no-build-isolation --no-deps -w /content/wheels . -v',
       env=env)
    import glob
    whl = glob.glob('/content/wheels/vllm-*.whl')[0]
    sh(f'{VPIP} install -q --force-reinstall --no-deps {whl}')
    print('source-built wheel installed:', whl)


In [ ]:
# Cell 4 - Hugging Face auth (google/gemma-* are gated: accept the license on the HF
# model page first). Prefers the Colab secret HF_TOKEN; falls back to the login widget.
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    ENV_BASE['HF_TOKEN'] = os.environ['HF_TOKEN']
    print('HF token loaded from Colab secret HF_TOKEN')
except Exception as e:
    print('no Colab HF_TOKEN secret (', e, ') - opening login widget')
    from huggingface_hub import notebook_login
    notebook_login()
    tok = None
    try:
        from huggingface_hub import HfFolder
        tok = HfFolder.get_token()
    except Exception:
        pass
    if tok:
        os.environ['HF_TOKEN'] = tok
        ENV_BASE['HF_TOKEN'] = tok


In [ ]:
# Cell 5 - serving helpers: launch `vllm serve`, wait-ready, chat smoke, KV-capacity
# extraction, quick PPL via the campaign harness. One server at a time.
import json, os, re, signal, subprocess, time, urllib.request

RESULTS_DIR = '/content/results_g4_gemma4'
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = []
PORT = 8000
_CUR = {'proc': None}

def stop_server():
    if _CUR['proc'] is not None and _CUR['proc'].poll() is None:
        _CUR['proc'].kill()
    _CUR['proc'] = None
    # vLLM renames its worker procs (setproctitle), so pkill-by-NAME misses the
    # EngineCore that actually holds the GPU. Kill whoever holds VRAM, by PID.
    def _gpu_pids():
        try:
            out = subprocess.run('nvidia-smi --query-compute-apps=pid --format=csv,noheader',
                                 shell=True, capture_output=True, text=True).stdout
            return [int(x) for x in out.split() if x.strip().isdigit()]
        except Exception:
            return []
    for _ in range(15):
        pids = _gpu_pids()
        if not pids:
            break
        for pid in pids:
            try: os.kill(pid, signal.SIGKILL)
            except Exception: pass
        time.sleep(2)
    subprocess.run('pkill -9 -f "vllm serve"; pkill -9 -f api_server', shell=True)  # belt+suspenders
    # confirm VRAM actually drained before returning
    for _ in range(15):
        try:
            free = int(subprocess.run('nvidia-smi --query-gpu=memory.free --format=csv,noheader,nounits',
                       shell=True, capture_output=True, text=True).stdout.split()[0])
            if free > 60000:
                break
        except Exception:
            pass
        time.sleep(2)

def launch(model, kv_mode, max_len, util, extra_args, extra_env, log_path):
    stop_server()
    env = dict(ENV_BASE, **extra_env)
    args = [f'{VENV}/bin/vllm', 'serve', model,
            '--max-model-len', str(max_len),
            '--gpu-memory-utilization', str(util),
            '--port', str(PORT)] + extra_args
    if kv_mode:
        args += ['--kv-cache-dtype', kv_mode]
    print('+', ' '.join(args))
    if extra_env:
        print('  env:', ' '.join(f'{k}={v}' for k, v in extra_env.items()))
    log = open(log_path, 'w')
    _CUR['proc'] = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, env=env)
    return _CUR['proc']

def wait_ready(proc, log_path, timeout_s=2700):
    # generous: first run includes the model download (31B bf16 ~62 GB) + FlashInfer JIT
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            urllib.request.urlopen(f'http://localhost:{PORT}/v1/models', timeout=2)
            return True
        except Exception:
            pass
        if proc.poll() is not None:
            print('SERVER EXITED, log tail:')
            print(open(log_path).read()[-2500:])
            return False
        time.sleep(5)
    print(f'not ready after {timeout_s}s, log tail:')
    print(open(log_path).read()[-2500:])
    return False

def chat(model, prompt, max_tok=96):
    body = json.dumps({'model': model, 'temperature': 0, 'max_tokens': max_tok,
                       'messages': [{'role': 'user', 'content': prompt}]}).encode()
    t0 = time.time()
    r = json.load(urllib.request.urlopen(urllib.request.Request(
        f'http://localhost:{PORT}/v1/chat/completions', body,
        {'Content-Type': 'application/json'}), timeout=300))
    return r['choices'][0]['message']['content'], time.time() - t0

def kv_capacity(log_text):
    m = re.search(r'GPU KV cache size:\s*([0-9,]+)\s*tokens', log_text)
    return int(m.group(1).replace(',', '')) if m else None

def proof_lines(log_text):
    pat = r'FLASHINFER|FLASH_ATTN|VO split|LINEAR_V_SF|GPU KV cache size|kv_cache_dtype|attention backend'
    return [l for l in log_text.splitlines() if re.search(pat, l)][-10:]

def quick_ppl(model, run_id, ctx=2048):
    out = f'{RESULTS_DIR}/{run_id}_ppl.json'
    r = subprocess.run(
        [VPY, f'{HIJINKS}/scripts/vllm_prompt_ppl_sweep.py',
         '--url', f'http://localhost:{PORT}', '--model', model, '--tokenizer', model,
         '--text-file', C1_CORPUS, '--ctx', str(ctx), '--run-id', run_id,
         '--runtime-ref', f'Colab G4 sm_120 notebook {NOTEBOOK_VERSION}',
         '--output', out],
        capture_output=True, text=True, env=ENV_BASE)
    if r.returncode != 0:
        print('PPL FAILED:', (r.stdout + r.stderr)[-1200:])
        return None
    j = json.load(open(out))
    return j['contexts'][0]['score']['mean_nll_nats']

def run_row(model, row_name, kv_mode, max_len, util, extra_args, extra_env,
            run_ppl=False, ppl_ctx=2048):
    tag = f"{model.split('/')[-1]}_{row_name}"
    log_path = f'{RESULTS_DIR}/{tag}_server.log'
    rec = {'model': model, 'row': row_name, 'notebook_version': NOTEBOOK_VERSION,
           'kv_mode': kv_mode or 'auto(bf16)', 'env_knobs': extra_env, 'ready': False,
           'kv_tokens': None, 'chat': None, 'ppl_mean_nll': None}
    try:
        proc = launch(model, kv_mode, max_len, util, extra_args, extra_env, log_path)
        rec['ready'] = wait_ready(proc, log_path)
        txt = open(log_path).read()
        rec['kv_tokens'] = kv_capacity(txt)
        rec['proof_lines'] = proof_lines(txt)
        for l in rec['proof_lines']:
            print('  |', l[-160:])
        if rec['ready']:
            reply, dt = chat(model, 'Reply with exactly: g4-ok. Then name the capital of Japan.')
            rec['chat'] = reply
            print(f'  chat ({dt:.1f}s): {reply[:160]!r}')
            if run_ppl:
                rec['ppl_mean_nll'] = quick_ppl(model, f'{tag}_c1_ctx{ppl_ctx}', ppl_ctx)
                print(f'  C1 ctx-{ppl_ctx} mean_nll_nats: {rec["ppl_mean_nll"]}')
    finally:
        stop_server()   # ALWAYS free the GPU after every row, even on exception
    RESULTS.append(rec)
    json.dump(rec, open(f'{RESULTS_DIR}/{tag}_{NOTEBOOK_VERSION}.json', 'w'), indent=2)
    print(f'=== {tag}: ready={rec["ready"]} kv_tokens={rec["kv_tokens"]} ===')
    return rec

def bench_serve(model, run_id, in_len=1024, out_len=256, num_prompts=160, conc=32):
    # vllm bench serve against the running server -> throughput + latency.
    out = f"{RESULTS_DIR}/{run_id}_bench.json"
    args = [f"{VENV}/bin/vllm", "bench", "serve", "--model", model,
            "--base-url", f"http://localhost:{PORT}", "--dataset-name", "random",
            "--num-prompts", str(num_prompts), "--random-input-len", str(in_len),
            "--random-output-len", str(out_len), "--max-concurrency", str(conc),
            "--ignore-eos", "--save-result", "--result-filename", out]
    r = subprocess.run(args, capture_output=True, text=True, env=ENV_BASE)
    if r.returncode != 0:
        print("BENCH FAILED:", (r.stdout + r.stderr)[-1500:]); return None
    j = json.load(open(out))
    keys = ("request_throughput", "output_throughput", "total_token_throughput",
            "mean_ttft_ms", "median_ttft_ms", "mean_tpot_ms", "median_tpot_ms", "mean_itl_ms")
    return {k: j.get(k) for k in keys}


## The A/B/C matrix — set `CELL` per runtime, then Run-All

Each cell forces its backend **explicitly** so the comparison is clean. Fast models first
(E4B/12B/1B/4B) so a partial run is already publishable; the big MoE/large models churn last.


In [ ]:
# ===================== SET THIS PER RUNTIME =====================
CELL = "A"            # "A" Triton/bf16  |  "B" FlashInfer/bf16  |  "C" FlashInfer/nvfp4
MODEL_FILTER = None   # e.g. ["google/gemma-4-12B-it"] to shard further; None = all that fit
BENCH = dict(in_len=1024, out_len=256, num_prompts=160, conc=32)
# ===============================================================
CELLS = {
 "A": dict(label="A_triton_bf16",      backend="TRITON_ATTN", kv=None,    env={}),
 "B": dict(label="B_flashinfer_bf16",  backend="FLASHINFER",  kv=None,    env={"VLLM_FLASHINFER_BF16_GEMMA": "1"}),
 "C": dict(label="C_flashinfer_nvfp4", backend="FLASHINFER",  kv="nvfp4", env={"VLLM_FLASHINFER_BF16_GEMMA": "1", "VLLM_NVFP4_KV_VOSPLIT": "1", "VLLM_NVFP4_KV_LINEAR_V_SF": "1"}),
}
assert CELL in CELLS, CELL
CFG = CELLS[CELL]
print("THIS RUNTIME RUNS CELL", CELL, "->", CFG["label"], "| backend", CFG["backend"], "| KV", CFG["kv"] or "bf16")

# all Gemma 3 + Gemma 4, fast-first order
ALL_BENCH_MODELS = [
 "google/gemma-4-E4B-it", "google/gemma-4-12B-it", "google/gemma-3-1b-it", "google/gemma-3-4b-it",
 "google/gemma-4-E2B-it", "google/gemma-3-12b-it", "google/gemma-3-27b-it",
 "google/gemma-4-26B-A4B-it", "google/gemma-4-31B-it",
]
BENCH_KNOBS = {
 "google/gemma-4-E2B-it":     dict(max_len=8192, util=0.85),
 "google/gemma-4-E4B-it":     dict(max_len=8192, util=0.85),
 "google/gemma-4-12B-it":     dict(max_len=8192, util=0.85),
 "google/gemma-4-26B-A4B-it": dict(max_len=8192, util=0.85),
 "google/gemma-4-31B-it":     dict(max_len=8192, util=0.90),
 "google/gemma-3-1b-it":      dict(max_len=8192, util=0.60),
 "google/gemma-3-4b-it":      dict(max_len=8192, util=0.80),
 "google/gemma-3-12b-it":     dict(max_len=8192, util=0.85),
 "google/gemma-3-27b-it":     dict(max_len=8192, util=0.90),
}
RUN_LIST = [m for m in ALL_BENCH_MODELS if (MODEL_FILTER is None or m in MODEL_FILTER)]
print("models for this runtime:", RUN_LIST)


In [ ]:
# Bench loop for THIS cell. One server at a time; GPU freed after every model.
def bench_model(model):
    knobs = BENCH_KNOBS[model]
    tag = model.split("/")[-1] + "_" + CFG["label"]
    log = f"{RESULTS_DIR}/{tag}_server.log"
    rec = {"model": model, "cell": CELL, "label": CFG["label"], "backend": CFG["backend"],
           "kv": CFG["kv"] or "bf16", "notebook_version": NOTEBOOK_VERSION, "ready": False,
           "kv_tokens": None, "chat": None, "bench": None}
    extra = ["--language-model-only", "--attention-backend", CFG["backend"]]
    try:
        proc = launch(model, CFG["kv"], knobs["max_len"], knobs["util"], extra, CFG["env"], log)
        rec["ready"] = wait_ready(proc, log)
        txt = open(log).read()
        rec["kv_tokens"] = kv_capacity(txt)
        rec["proof_lines"] = proof_lines(txt)
        for l in rec["proof_lines"]:
            print("  |", l[-150:])
        if rec["ready"]:
            reply, dt = chat(model, "Reply with exactly: ok. Then name the capital of Japan.")
            rec["chat"] = reply
            print("  chat ({:.1f}s): {!r}".format(dt, reply[:90]))
            rec["bench"] = bench_serve(model, tag, **BENCH)
            print("  bench:", rec["bench"])
    finally:
        stop_server()
    RESULTS.append(rec)
    json.dump(rec, open(f"{RESULTS_DIR}/{tag}.json", "w"), indent=2)
    tput = rec["bench"].get("output_throughput") if rec["bench"] else None
    print("=== {}: ready={} kv={} out_tok/s={} ===".format(tag, rec["ready"], rec["kv_tokens"], tput))
    return rec

for _m in RUN_LIST:
    bench_model(_m)


In [ ]:
# Per-cell summary + save to the SHARED Drive folder so A/B/C converge.
import json, shutil
SAVE_TO_DRIVE = True
DRIVE_SUBDIR = "gemma_retirement_bench"
print("=== CELL {} ({}) - notebook {} ===".format(CELL, CFG["label"], NOTEBOOK_VERSION))
hdr = "{:26} {:5} {:>11} {:>10} {:>9} {:>9}  chat".format("model", "ok", "KV tok", "out tok/s", "TTFT ms", "TPOT ms")
print(hdr); print("-" * len(hdr))
for r in RESULTS:
    b = r["bench"] or {}
    kv = "{:,}".format(r["kv_tokens"]) if r["kv_tokens"] else "-"
    tput = "{:.0f}".format(b["output_throughput"]) if b.get("output_throughput") else "-"
    ttft = "{:.0f}".format(b["mean_ttft_ms"]) if b.get("mean_ttft_ms") else "-"
    tpot = "{:.1f}".format(b["mean_tpot_ms"]) if b.get("mean_tpot_ms") else "-"
    print("{:26} {:5} {:>11} {:>10} {:>9} {:>9}  {}".format(
          r["model"].split("/")[-1], str(r["ready"]), kv, tput, ttft, tpot, (r["chat"] or "")[:28]))
json.dump(RESULTS, open(f"{RESULTS_DIR}/SUMMARY_{CFG[label]}.json", "w"), indent=2)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/{DRIVE_SUBDIR}/{CFG[label]}"
    shutil.copytree(RESULTS_DIR, dst, dirs_exist_ok=True)
    print("copied ->", dst)
print()
print("NOTE: Triton/nvfp4 is N/A by construction - Triton has no NVFP4 KV path.")
print("That empty cell IS the necessity result: shipping nvfp4 KV requires FlashInfer.")
